<a href="https://colab.research.google.com/github/LoPA607/IE643/blob/main/PlainAdain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image



In [ ]:
# -----------------------------
# 1. Device
# -----------------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# -----------------------------
# 2. Image loading
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

content_img = Image.open('/content/dog.jpg').convert('RGB')
style_img = Image.open('/content/Screenshot from 2025-10-04 23-08-58.png').convert('RGB')

content = transform(content_img).unsqueeze(0).to(device)
style = transform(style_img).unsqueeze(0).to(device)


In [ ]:
# -----------------------------
# 3. Pretrained VGG Encoder
# -----------------------------
vgg_pretrained = models.vgg19(pretrained=True).features.to(device).eval()
for param in vgg_pretrained.parameters():
    param.requires_grad = False

class VGGEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = {'3':'relu1_2','8':'relu2_2','17':'relu3_4','26':'relu4_4'}
        self.vgg = vgg_pretrained

    def forward(self, x):
        features = {}
        for name, layer in self.vgg._modules.items():
            x = layer(x)
            if name in self.layers:
                features[self.layers[name]] = x
        return features

encoder = VGGEncoder().to(device)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:05<00:00, 105MB/s] 


In [ ]:
# -----------------------------
# 4. Decoder (trained from scratch)
# -----------------------------
class Decoder(nn.Module):
    def __init__(self, latent_dim=512):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(latent_dim, 256, 3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2), # 32x32 -> 64x64
            nn.Conv2d(256, 128, 3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2), # 64x64 -> 128x128
            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2), # 128x128 -> 256x256
            nn.Conv2d(64, 3, 3, padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

decoder = Decoder().to(device)

In [ ]:
# -----------------------------
# 5. AdaIN Function
# -----------------------------
def adain(content_feat, style_feat, eps=1e-5):
    c_mean, c_std = content_feat.mean([2,3], keepdim=True), content_feat.std([2,3], keepdim=True)
    s_mean, s_std = style_feat.mean([2,3], keepdim=True), style_feat.std([2,3], keepdim=True)
    return s_std * (content_feat - c_mean) / (c_std + eps) + s_mean


In [ ]:
# -----------------------------
# 6. Loss Functions
# -----------------------------
def gram_matrix(feat):
    b, c, h, w = feat.size()
    feat = feat.view(b, c, h*w)
    gram = torch.bmm(feat, feat.transpose(1,2)) / (c*h*w)
    return gram

def compute_loss(out, content, style, content_feat): # Added content_feat as an argument
    out_feats = encoder(out)
    style_feats = encoder(style)

    # Content loss
    loss_content = F.mse_loss(out_feats['relu4_4'], content_feat) # Changed target to content_feat

    # Style loss
    loss_style = 0
    for layer in ['relu1_2','relu2_2','relu3_4','relu4_4']:
        out_gram = gram_matrix(out_feats[layer])
        style_gram = gram_matrix(style_feats[layer])
        loss_style += F.mse_loss(out_gram, style_gram)

    return loss_content, loss_style

In [ ]:
# -----------------------------
# 7. Optimizer
# -----------------------------
optimizer = torch.optim.Adam(decoder.parameters(), lr=1e-4)

# -----------------------------
# 8. Training Loop (one-shot)
# -----------------------------
num_epochs = 15000  # since it's one-shot, can train longer
for epoch in range(num_epochs):
    decoder.train()

    # Encode content and style
    content_feat = encoder(content)['relu4_4']
    style_feat = encoder(style)['relu4_4']

    # AdaIN
    t = adain(content_feat, style_feat)

    # Decode
    out = decoder(t)

    # Compute loss
    loss_c, loss_s = compute_loss(out, content, style, content_feat)
    loss = 1.0*loss_c + 1000000.0*loss_s

    # Backprop
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 200 == 0:
        print(f'Epoch {epoch+1}, Content Loss: {loss_c.item():.4f}, Style Loss: {loss_s.item():.4f}')
        print(f'Total Loss: {loss.item():.4f}')

# -----------------------------
# 9. Save output
# -----------------------------
out_img = out.detach().cpu().squeeze(0)
out_img = transforms.ToPILImage()( (out_img + 1)/2 )  # rescale [-1,1] to [0,1]
out_img.save('output.png')
print("Stylized image saved as output.png")

Epoch 200, Content Loss: 0.7008, Style Loss: 0.0000
Total Loss: 0.9072
Epoch 400, Content Loss: 0.5493, Style Loss: 0.0000
Total Loss: 0.6621
Epoch 600, Content Loss: 0.4805, Style Loss: 0.0000
Total Loss: 0.5876
Epoch 800, Content Loss: 0.4452, Style Loss: 0.0000
Total Loss: 0.5495
Epoch 1000, Content Loss: 0.4202, Style Loss: 0.0000
Total Loss: 0.5255
Epoch 1200, Content Loss: 0.4000, Style Loss: 0.0000
Total Loss: 0.5049
Epoch 1400, Content Loss: 0.3839, Style Loss: 0.0000
Total Loss: 0.4887
Epoch 1600, Content Loss: 0.3717, Style Loss: 0.0000
Total Loss: 0.4765
Epoch 1800, Content Loss: 0.3632, Style Loss: 0.0000
Total Loss: 0.4701
Epoch 2000, Content Loss: 0.3535, Style Loss: 0.0000
Total Loss: 0.4597
Epoch 2200, Content Loss: 0.3468, Style Loss: 0.0000
Total Loss: 0.4518
Epoch 2400, Content Loss: 0.3401, Style Loss: 0.0000
Total Loss: 0.4456
Epoch 2600, Content Loss: 0.3339, Style Loss: 0.0000
Total Loss: 0.4395
Epoch 2800, Content Loss: 0.3291, Style Loss: 0.0000
Total Loss: 0.4